In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)

## Shared data and NumPy reference

In [ ]:
X_np, y_np = make_moons(n_samples=300, noise=0.12, random_state=SEED)
X_np = X_np.astype(np.float64); y_np = y_np.astype(np.float64).reshape(-1,1)
X = torch.tensor(X_np); y = torch.tensor(y_np)

def numpy_init(input_dim, hidden_dim):
    rng = np.random.default_rng(SEED + hidden_dim)
    return {"W1": rng.normal(0,0.15,(input_dim,hidden_dim)), "b1": np.zeros((1,hidden_dim)), "W2": rng.normal(0,0.15,(hidden_dim,1)), "b2": np.zeros((1,1))}

def numpy_forward(p, X):
    z1=X@p["W1"]+p["b1"]; a1=np.maximum(z1,0); return 1/(1+np.exp(-np.clip(a1@p["W2"]+p["b2"],-50,50)))

def numpy_loss_grads(p,X,y):
    z1=X@p["W1"]+p["b1"]; a1=np.maximum(z1,0); yh=numpy_forward(p,X); m=len(X); dz2=yh-y
    g={"W2":a1.T@dz2/m,"b2":dz2.mean(0,keepdims=True)}; dz1=(dz2@p["W2"].T)*(z1>0); g["W1"]=X.T@dz1/m; g["b1"]=dz1.mean(0,keepdims=True)
    loss=-np.mean(y*np.log(yh+1e-12)+(1-y)*np.log(1-yh+1e-12))
    return loss,g

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden):
        super().__init__(); self.fc1=nn.Linear(2,hidden); self.relu=nn.ReLU(); self.fc2=nn.Linear(hidden,1)
    def forward(self,x): return torch.sigmoid(self.fc2(self.relu(self.fc1(x))))

def copy_numpy_weights(model, p):
    with torch.no_grad():
        model.fc1.weight.copy_(torch.tensor(p["W1"].T)); model.fc1.bias.copy_(torch.tensor(p["b1"].ravel()))
        model.fc2.weight.copy_(torch.tensor(p["W2"].T)); model.fc2.bias.copy_(torch.tensor(p["b2"].ravel()))

## Full-batch SGD: NumPy and PyTorch should match


In [ ]:
hidden=8; epochs=300; lr=0.25
p = numpy_init(2,hidden); model=MLP(hidden); copy_numpy_weights(model,p)
loss_fn=nn.BCELoss(); opt=torch.optim.SGD(model.parameters(),lr=lr)
np_losses=[]; torch_losses=[]
for epoch in range(epochs):
    loss_np,g=numpy_loss_grads(p,X_np,y_np); np_losses.append(loss_np)
    for key in p: p[key]-=lr*g[key]
    opt.zero_grad(); out=model(X); loss=loss_fn(out,y); loss.backward(); opt.step(); torch_losses.append(loss.item())
max_loss_diff=max(abs(a-b) for a,b in zip(np_losses,torch_losses))
print("max per-epoch loss difference:",max_loss_diff)
print("final prediction max difference:",np.max(np.abs(numpy_forward(p,X_np)-model(X).detach().numpy())))
assert max_loss_diff < 1e-5

In [ ]:
# Gradient display at a fresh, identical first step
p0=numpy_init(2,hidden); m0=MLP(hidden); copy_numpy_weights(m0,p0); out=m0(X); loss=loss_fn(out,y); loss.backward(); _,g0=numpy_loss_grads(p0,X_np,y_np)
print("manual dW1 first row:",g0["W1"][0]); print("torch dW1 first row:",m0.fc1.weight.grad.detach().numpy().T[0])
print("manual dW2:",g0["W2"].ravel()[:4]); print("torch dW2:",m0.fc2.weight.grad.detach().numpy().ravel()[:4])

## Normal PyTorch training: Adam and mini-batches


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(SEED); loader=DataLoader(TensorDataset(X,y),batch_size=32,shuffle=True)
adam_model=MLP(8); adam= torch.optim.Adam(adam_model.parameters(),lr=0.01); adam_losses=[]
for _ in range(100):
    running=0.0
    for xb,yb in loader:
        adam.zero_grad(); l=loss_fn(adam_model(xb),yb); l.backward(); adam.step(); running += l.item()*len(xb)
    adam_losses.append(running/len(X))
plt.plot(np_losses,label="NumPy SGD"); plt.plot(torch_losses,label="PyTorch SGD"); plt.plot(np.linspace(0,epochs-1,100),adam_losses,label="Adam mini-batch"); plt.xlabel("epoch"); plt.ylabel("BCE loss"); plt.legend(); plt.title("Controlled match and normal training"); plt.show()


Autograd reproduced the manual chain rule and exposed gradients for each parameter without requiring separate derivative code for every layer. The controlled comparison is still valuable because it proves the model, dtype, initialization, and update order match rather than merely producing a plausible curve.